In [ ]:
import sys
from pathlib import Path
import torch

from tracers.project import setup_project
from tracers.datasets.factory import build_datasets, build_loader_for_model
from tracers.pipelines.trainer_factory import build_trainer

PROJECT_ROOT = Path.cwd().parent
setup = setup_project(PROJECT_ROOT)

MODEL_KEY = "tracer_global_np"

device = setup["device"]
experiment_config = setup["experiment_config"]
hyperparams = setup["hyperparams"]
segment_frames = setup["segment_frames"]
batch_size = setup["batch_size"]

In [ ]:
audio_dir, beatmap_dir, train_dataset, val_dataset, test_dataset, train_ids, val_ids, test_ids = build_datasets(
    PROJECT_ROOT,
    segment_frames=segment_frames,
    seed=42,
)

train_loader = build_loader_for_model(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    model_key=MODEL_KEY,
)
val_loader = build_loader_for_model(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)
test_loader = build_loader_for_model(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)

cuda


In [ ]:
trainer, model = build_trainer(
    model_key=MODEL_KEY,
    hyperparams=hyperparams,
    experiment_config=experiment_config,
    device=device,
    checkpoint_path=(
        str(PROJECT_ROOT / experiment_config.resume_from)
        if experiment_config.resume_from is not None
        else None
    ),
    train_loader=train_loader,
    valid_loader=val_loader,
)

# Load checkpoint if specified in YAML
if experiment_config.resume_from is not None:
    resume_from = PROJECT_ROOT / experiment_config.resume_from
    trainer.load_checkpoint(resume_from)
    for g in trainer.optimizer.param_groups:
        g["lr"] = hyperparams["learning_rate"]

    print("learning_rate set to ", hyperparams["learning_rate"])

In [9]:
trainer.fit()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\joshua\_netrc.
wandb: Currently logged in as: joshuardh (joshuardh-mindanao-state-university) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Training: 100%|██████████| 300/300 [07:16<00:00,  1.46s/it]
c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\.venv\Lib\site-packages\torch\nn\modules\transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(
Validation: 100%|██████████| 151/151 [02:08<00:00,  1.17it/s]



Epoch 3/40

TRAIN
Loss   | Total: 0.2711 | NP: 0.0180 | NS: 0.1450
Onset  | P: 0.7715 | R: 0.6329 | F1: 0.6954
Action | P: 0.8973 | R: 0.5803 | F1: 0.7048
Active Exact Accuracy: 0.0926
GT Active Rate: 5.01% | Predicted Active Rate: 3.24%

VAL
Loss   | Total: 0.2441 | NP: 0.0169 | NS: 0.1256
Onset  | P: 0.7842 | R: 0.5796 | F1: 0.6665
Action | P: 0.8816 | R: 0.6141 | F1: 0.7239
Active Exact Accuracy: 0.1021
GT Active Rate: 4.37% | Predicted Active Rate: 3.04%


Epochs:   3%|▎         | 1/38 [09:26<5:49:09, 566.20s/it]

Saved checkpoint: checkpoints\latest.pt
Saved checkpoint: checkpoints\best.pt


Validation: 100%|██████████| 151/151 [02:08<00:00,  1.17it/s]


... (110 outputs left out)

Saved checkpoint: checkpoints\best.pt


Epochs: 100%|██████████| 38/38 [5:52:42<00:00, 556.91s/it]



Epoch 40/40

TRAIN
Loss   | Total: 0.1276 | NP: 0.0097 | NS: 0.0598
Onset  | P: 0.8246 | R: 0.8268 | F1: 0.8257
Action | P: 0.9048 | R: 0.8780 | F1: 0.8912
Active Exact Accuracy: 0.3997
GT Active Rate: 4.97% | Predicted Active Rate: 4.83%

VAL
Loss   | Total: 0.1521 | NP: 0.0128 | NS: 0.0626
Onset  | P: 0.7327 | R: 0.7797 | F1: 0.7554
Action | P: 0.8471 | R: 0.8687 | F1: 0.8578
Active Exact Accuracy: 0.3737
GT Active Rate: 4.37% | Predicted Active Rate: 4.48%
Saved checkpoint: checkpoints\latest.pt


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
time/epoch_seconds,█▅▅▄▄▅▂▇▃▅▅▃▅▅▄▃▅▃▄▅▄▅▄▄▄▁▄▅▆▄▆▂▂▃▃▂▆▇
time/train_seconds,█▆▅▅▅▅▃▇▄▅▅▃▅▅▄▃▄▃▄▄▄▄▄▄▄▁▄▅▆▄▆▃▂▃▃▂▅▆
time/val_seconds,▄▄▆▂▃▄▁▃▃▅▅▅▆▄▅▅█▅▅▆▆▆▆▆▄▆▅▅▅▆▅▃▆▆▅▇█▆
train/action_f1,▁▂▂▃▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████████
train/action_precision,▃▇▇▇▂▁▁▁▂▁▂▃▂▃▂▁▃▁▂▂▂▂▂▄▃▄▃▅▄▅▅▅▆▆▆▇█▇
train/action_recall,▁▂▂▃▃▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇█████████████
train/active_exact_accuracy,▁▁▂▂▃▃▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇████████
train/gt_active_rate,▅▄▄▄▅▅▆▄▆▄▄▆▁▆▄▅▆▂▅▇▄▇▂▃▃▄▆▅▄▂▄▄▃█▅▃▄▂
train/np_loss,█▇▇▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
+18,...
